# Week 1 briefing: embeddings, Overture GT, US baselines

AlphaEarth Satellite Embedding V1 is already global at 10 m (2017-2024). Overture use tags are schema-clear where present, but fill swings from about 75% in Berlin to about 1% in Lagos. On a balanced US binary sample, logistic regression on 64-D centroids hits about 0.77 mean LORO macro-F1. Fine 10-class typing on the same embeddings is only about 0.22. Labels and unit choice matter more than waiting on a custom embedding run.

Setup: Overture buildings + 2024 AlphaEarth centroids, leave-one-region-out (LORO). Binary trial uses 9 metros × 400 buildings. Multi-class Protocol B uses about 11.3k buildings × 10 metros × 10 types. Figures under `outputs/`; no retrain needed to read this notebook.


## 0. What we needed before modeling

Two inputs for the carbon / building-use study: a global embedding field, and building-use labels. Embeddings are not the blocker. Label fill and how we treat unlabeled footprints are.

| Input | Status |
|---|---|
| AlphaEarth V1 annual | Global land + shallow coast; EE / GCS; 64-D; weights not released |
| Overture `subtype` / `class` | Clear taxonomy; OSM-derived fill; ML footprints usually unlabeled |
| Unlabeled buildings | Excluded from GT; never treated as non-residential |

Definition lock: [`docs/residential_definition_v1.md`](../../docs/residential_definition_v1.md) (`resi_def_v1`).


## 1. Embeddings: global access up front

Public layer `GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL`. Same representation in Boston, Berlin, or Lagos. Custom Maps Platform embeddings stay optional for later sub-annual windows.

- Resolution: 10 m (a small house is about 1-4 cells; neighborhood context is in the training chips)
- Years: 2017-2024
- Sensors in the foundation stack: optical + Sentinel-1 SAR and related streams (not a façade VHR product)

Supporting notebook: `notebooks/01_phase1_alphaearth_overture_briefing.ipynb`.


## 2. Overture GT: clear categories, uneven fill

Live extracts (public GeoParquet) showed subtype fill roughly:

| City | Approx. % with subtype |
|---|---:|
| Berlin | ~75% |
| Amsterdam | ~70% |
| Boston | ~22% |
| Houston | ~16% |
| Nairobi | ~2% |
| Lagos | ~1% |

Residential vs other non-null subtypes is usable when a tag exists. SF vs MF needs high-precision `class` (`house` / `detached` vs `apartments`). Weak tracks (subtype-only, Places, GHS-OBAT) stay out of Track-A F1.

GT stack we proposed and kept:

1. Clean binary / high-precision class where tagged
2. Noisy automated labels to expand coverage later
3. Unlabeled footprints stay out of training negatives


## 3. Binary US trial (centroid LR)

Balanced RES / NRES sample, logistic regression on 64-D AlphaEarth, LORO across 9 metros.

- Pooled 80/20 holdout macro-F1 about 0.83 (optimistic)
- LORO mean macro-F1 about 0.77 (primary)
- Holdout range: Atlanta about 0.87 … NYC about 0.57

![US region performance map](../../outputs/trial_residential/us_region_performance_map.png)

![US region performance bars](../../outputs/trial_residential/us_region_performance_bars.png)

![Pooled vs LORO ROC / PR](../../outputs/trial_residential/roc_pr_pooled_vs_loro.png)

![ROC / PR by region](../../outputs/trial_residential/roc_pr_loro_by_region.png)

Notebook: `notebooks/02_trial_residential_classifier.ipynb`. Snapshot: `outputs/trial_residential/trial_snapshot.json`.


## 4. Multi-class floor (10 types, AlphaEarth only)

Protocol B on the SOTA-comparable US set (about 11.3k buildings, 10 metros, 10 types). Centroid features only; not modality-matched to street-view / VHR papers.

| Protocol | Best model | Macro-F1 |
|---|---|---:|
| LORO (primary) | HistGradientBoosting | 0.22 ± 0.03 |
| Pooled 80/20 | HistGradientBoosting | ~0.40 |
| Chance / majority | - | ~0.10 |

![LORO macro-F1 by region](../../outputs/us_loro_baseline/loro_macro_f1_by_region.png)

![US LORO macro-F1 map](../../outputs/us_loro_baseline/loro_us_macro_f1_map.png)

![Per-class F1](../../outputs/us_loro_baseline/loro_per_class_f1.png)

![HGB confusion (OOF)](../../outputs/us_loro_baseline/loro_confusion_hgb.png)

Takeaway for the room: embeddings carry some type signal and support a global method. They are not enough alone, at centroid sampling, for fine geographically robust typing. First carbon-relevant win stays binary residential plus coarse archetypes.

Notebooks: `04_sota_comparable_building_type_dataset.ipynb`, `05_us_loro_building_type_baseline.ipynb`.


## 5. Takeaways

| Decision | Evidence |
|---|---|
| Use Annual AlphaEarth V1 via EE for Phases 1-3 | Global 10 m coverage already shipped |
| Lock `resi_def_v1` for Track-A GT | Class-based SF/MF; unlabeled excluded |
| Keep LORO as the primary metric | Pooled scores overstate transfer |
| Prefer binary RES/NRES before fine types | 0.77 binary LORO vs 0.22 ten-class |
| Plan noisy GT for OSM-poor cities | Fill can fall to ~1%; never invent negatives |

Related: `docs/findings.md`, `docs/residential_definition_v1.md`, action items from Meeting 1.


## Appendix: load live numbers


In [ ]:
import json
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path('.').resolve()
for _ in range(3):
    if (ROOT / 'requirements.txt').exists():
        break
    ROOT = ROOT.parent

paths = {
    'binary trial': ROOT / 'outputs/trial_residential/trial_snapshot.json',
    '10-class LORO baseline': ROOT / 'outputs/us_loro_baseline/baseline_snapshot.json',
}
for name, p in paths.items():
    display(Markdown(f'### {name}'))
    if not p.exists():
        print('missing:', p)
        continue
    d = json.loads(p.read_text())
    if 'loro_macro_f1_mean' in d:
        keep = {
            'n_buildings': d.get('n_buildings'),
            'loro_macro_f1_mean': d.get('loro_macro_f1_mean'),
            'loro_accuracy_mean': d.get('loro_accuracy_mean'),
            'pooled_holdout': d.get('pooled_holdout'),
        }
    else:
        keep = {
            'n_buildings': d.get('n_buildings'),
            'best_model_by_loro_macro_f1': d.get('best_model_by_loro_macro_f1'),
            'loro_summary': d.get('loro_summary'),
            'pooled_holdout': d.get('pooled_holdout'),
        }
    print(json.dumps(keep, indent=2)[:2500])
